# Modulo 8 · Lezione 7 — Reti Convoluzionali (CNN): Visione Artificiale

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Perché le reti dense falliscono con le immagini |
| 2 | La convoluzione: rilevare pattern locali |
| 3 | Pooling: ridurre la dimensione |
| 4 | Architettura CNN completa in Keras |
| 5 | Transfer Learning: riutilizzare reti preaddestrate |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.signal import convolve2d

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow: {tf.__version__}")

---
## 1. Perché le reti dense falliscono con le immagini

```
Un'immagine di 224×224×3 pixel (RGB) ha 150.528 valori.

Rete densa: 150.528 → 1.000 neuroni = 150.528.000 parametri solo nel primo strato!
Problemi:
  1. Troppi parametri → overfitting, memoria, lentezza
  2. Ignora la struttura spaziale: non sa che pixel vicini sono correlati
  3. Non è invariante alla traslazione: un 'gatto' nell'angolo ≠ 'gatto' al centro

Le CNN sfruttano tre proprietà delle immagini:
  1. Località: un occhio è un gruppo di pixel vicini
  2. Gerarchia: bordi → forme → oggetti
  3. Traslazione: lo stesso rilevatore di bordi funziona in tutta l'immagine

Soluzione: condividere i pesi tramite filtri convoluzionali
  Lo stesso filtro applicato in ogni posizione → molti meno parametri
```

In [ ]:
# Confrontare i parametri: Dense vs CNN per un'immagine 28×28
print("Confronto dei parametri: immagine 28×28 (MNIST)")
print("="*55)

# Rete densa
m_dense = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
print(f"Rete Densa:")
m_dense.summary()

print("\nRete CNN:")
# CNN
m_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),          # immagine 2D + canale
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
m_cnn.summary()

---
## 2. La convoluzione: rilevare pattern locali

```
Convoluzione 2D:
  Filtro (kernel): piccola matrice, ad esempio 3×3
  Scorre sull'immagine calcolando il prodotto scalare in ogni posizione
  Uscita: feature map → evidenzia dove è stato rilevato il pattern del filtro

  Immagine 28×28 * Filtro 3×3 → Feature map 26×26 (senza padding)
  Con padding='same' → mantiene 28×28

Conv2D(32, (3,3)):
  32 filtri diversi, ciascuno 3×3
  Parametri: 3×3×n_canali_ingresso × 32 + 32 (bias)
  Per ingresso RGB: 3×3×3×32 + 32 = 896 params
  Per MNIST (1 canale): 3×3×1×32 + 32 = 320 params

Cosa rilevano i filtri?
  Strato 1: bordi orizzontali, verticali, diagonali
  Strato 2: curve, angoli, texture
  Strato 3+: parti di oggetti, oggetti completi
```

In [ ]:
# Filtri classici per rilevare i bordi
filtros = {
    'Orizzontale (Sobel)': np.array([[-1,-2,-1], [0,0,0], [1,2,1]]),
    'Verticale (Sobel)':   np.array([[-1,0,1],  [-2,0,2], [-1,0,1]]),
    'Diagonale':           np.array([[2,1,-1],  [1,0,-1], [-1,-1,2]]),
    'Nitidezza':           np.array([[0,-1,0],  [-1,5,-1], [0,-1,0]]),
    'Blur':                np.ones((3,3)) / 9,
}

# Caricare un'immagine di MNIST come esempio
(X_tr, _), _ = keras.datasets.mnist.load_data()
img = X_tr[0].astype('float32') / 255.0  # 28×28

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0,0].imshow(img, cmap='gray'); axes[0,0].set_title('Originale'); axes[0,0].axis('off')

for ax, (nombre, f) in zip(axes.ravel()[1:], filtros.items()):
    feature_map = convolve2d(img, f, mode='same')
    ax.imshow(feature_map, cmap='RdBu_r')
    ax.set_title(f'Filtro: {nombre}', fontsize=9)
    ax.axis('off')

plt.suptitle('Convoluzioni con filtri artigianali', fontsize=14)
plt.tight_layout(); plt.show()
print("→ Una CNN impara questi filtri automaticamente dai dati")

In [ ]:
# Conv2D in Keras: parametri spiegati
print("""
layers.Conv2D(
    filters=32,           # numero di filtri (feature map di uscita)
    kernel_size=(3,3),    # dimensione del filtro. Può essere solo 3 (quadrato)
    strides=(1,1),        # passo dello scorrimento (default)
    padding='valid',      # 'valid' = senza padding, 'same' = mantiene la dimensione
    activation='relu',    # attivazione applicata al feature map
    use_bias=True,        # bias per filtro
    input_shape=(H,W,C)   # nel primo strato: altezza, larghezza, canali
)

Uscita con padding='valid':
  (H - k + 1) × (W - k + 1) × filters
  28×28 → Conv2D(32, 3) → 26×26×32

Uscita con padding='same':
  H × W × filters
  28×28 → Conv2D(32, 3, padding='same') → 28×28×32

Strides:
  Conv2D(32, 3, strides=2) → uscita ≈ H/2 × W/2 × 32 (riduce la risoluzione)
""")

# Calcolare i parametri di uno strato Conv2D
for c_in, n_filtros, k in [(1, 32, 3), (32, 64, 3), (3, 128, 3)]:
    params = (k*k*c_in + 1) * n_filtros
    print(f"Conv2D({n_filtros}, {k}×{k}) con {c_in} canale/i in ingresso: {params:,} parametri")

---
## 3. Pooling: ridurre la dimensione

In [ ]:
print("""
MaxPooling2D(pool_size=(2,2), strides=None):
  Divide il feature map in blocchi 2×2 e prende il massimo di ogni blocco.
  Riduce la risoluzione a metà: 26×26 → 13×13
  Vantaggio: invarianza a piccole traslazioni
  Parametri: 0 (non ci sono parametri da imparare)

AveragePooling2D: media invece del massimo
GlobalAveragePooling2D: media su TUTTA la dimensione spaziale → 1 valore per canale
  Utile nell'ultimo strato convoluzionale prima di Dense
""")

# Visualizzare MaxPooling
np.random.seed(0)
feat_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 1, 2],
    [1, 2, 8, 9],
    [3, 4, 6, 7]
], dtype=float)

pool = np.array([
    [max(feat_map[0,0], feat_map[0,1], feat_map[1,0], feat_map[1,1]),
     max(feat_map[0,2], feat_map[0,3], feat_map[1,2], feat_map[1,3])],
    [max(feat_map[2,0], feat_map[2,1], feat_map[3,0], feat_map[3,1]),
     max(feat_map[2,2], feat_map[2,3], feat_map[3,2], feat_map[3,3])],
])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im1 = axes[0].imshow(feat_map, cmap='Blues', vmin=0)
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, int(feat_map[i,j]), ha='center', va='center', fontsize=16)
# segnare i blocchi
for i in range(2):
    for j in range(2):
        rect = plt.Rectangle((j*2-0.5, i*2-0.5), 2, 2,
                              linewidth=3, edgecolor='red', facecolor='none')
        axes[0].add_patch(rect)
axes[0].set_title('Feature map 4×4')

im2 = axes[1].imshow(pool, cmap='Blues', vmin=0)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, int(pool[i,j]), ha='center', va='center',
                     fontsize=22, fontweight='bold')
axes[1].set_title('Dopo MaxPooling(2×2) → 2×2')

plt.suptitle('Max Pooling: prende il massimo in ogni blocco 2×2', fontsize=12)
plt.tight_layout(); plt.show()

---
## 4. Architettura CNN completa in Keras

In [ ]:
# CNN per MNIST
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()

# Le CNN hanno bisogno di shape (H, W, C) — aggiungere la dimensione del canale
X_tr_cnn = X_tr.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_te_cnn = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(f"Shape per CNN: {X_tr_cnn.shape}  (N, H, W, C)")

tf.random.set_seed(42)
modelo_cnn = keras.Sequential([
    # Blocco conv 1: rilevare pattern base
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),  # 28×28×32
    layers.Conv2D(32, 3, activation='relu'),                   # 26×26×32
    layers.MaxPooling2D(2),                                    # 13×13×32
    layers.Dropout(0.25),

    # Blocco conv 2: rilevare pattern composti
    layers.Conv2D(64, 3, activation='relu', padding='same'),  # 13×13×64
    layers.Conv2D(64, 3, activation='relu'),                   # 11×11×64
    layers.MaxPooling2D(2),                                    # 5×5×64
    layers.Dropout(0.25),

    # Classificatore
    layers.Flatten(),                                          # 5×5×64 = 1600
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
], name='CNN_MNIST')

modelo_cnn.summary()

In [ ]:
modelo_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

h_cnn = modelo_cnn.fit(
    X_tr_cnn, y_tr,
    epochs=12, batch_size=128,
    validation_split=0.1,
    verbose=1
)

_, acc_cnn = modelo_cnn.evaluate(X_te_cnn, y_te, verbose=0)
print(f"\nCNN — Test accuracy: {acc_cnn:.4f} ({acc_cnn:.2%})")

# Confrontare con Dense della lezione precedente (~97.5%)
# CNN tipica: >99%

In [ ]:
# Visualizzare i filtri appresi dal primo strato
W_conv = modelo_cnn.layers[0].get_weights()[0]  # (3, 3, 1, 32)
print(f"Filtri dello strato Conv2D(32): {W_conv.shape}")

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    filtro = W_conv[:, :, 0, i]  # filtro i
    ax.imshow(filtro, cmap='RdBu_r')
    ax.axis('off')
plt.suptitle('32 filtri appresi dal 1° strato Conv2D\n(blu=attiva, rosso=inibisce)', fontsize=12)
plt.tight_layout(); plt.show()

# Visualizzare i feature map per un'immagine
modelo_conv1 = keras.Model(inputs=modelo_cnn.input, outputs=modelo_cnn.layers[0].output)
img = X_te_cnn[0:1]  # una cifra
feature_maps = modelo_conv1.predict(img, verbose=0)  # (1, 28, 28, 32)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.axis('off')
plt.suptitle(f'Feature map (uscita strato Conv1) per la cifra {y_te[0]}', fontsize=12)
plt.tight_layout(); plt.show()

---
## 5. Transfer Learning: riutilizzare reti preaddestrate

```
Problema: addestrare una CNN grande da zero richiede:
  Milioni di immagini + settimane di calcolo su GPU

Transfer Learning: usare una rete già addestrata su ImageNet (1.2M immagini, 1000 classi)
  Gli strati convoluzionali imparano rilevatori di bordi, texture, forme...
  Questi rilevatori sono UTILI per qualsiasi compito di visione

Strategie:
  1. Feature extraction: congela gli strati conv, addestra solo il classificatore finale
  2. Fine-tuning: scongela gli ultimi strati conv e addestra con lr molto basso

Modelli disponibili in Keras:
  keras.applications.VGG16
  keras.applications.ResNet50
  keras.applications.MobileNetV2    ← leggero, ideale per la produzione
  keras.applications.EfficientNetB0 ← equilibrio prestazioni/velocità
  keras.applications.InceptionV3
```

In [ ]:
# Transfer Learning con MobileNetV2 per classificazione binaria
# Usiamo CIFAR-10: cani (classe 5) vs gatti (classe 3)
(X_tr_c, y_tr_c), (X_te_c, y_te_c) = keras.datasets.cifar10.load_data()

# Filtrare solo cani e gatti
mask_tr = (y_tr_c.ravel() == 3) | (y_tr_c.ravel() == 5)
mask_te = (y_te_c.ravel() == 3) | (y_te_c.ravel() == 5)

X_tr_gc = X_tr_c[mask_tr].astype('float32') / 255.0
y_tr_gc = (y_tr_c[mask_tr].ravel() == 5).astype('float32')  # 1=cane, 0=gatto
X_te_gc = X_te_c[mask_te].astype('float32') / 255.0
y_te_gc = (y_te_c[mask_te].ravel() == 5).astype('float32')

print(f"Cani vs Gatti — Train: {X_tr_gc.shape}, Test: {X_te_gc.shape}")
print(f"Classi: {np.bincount(y_tr_gc.astype(int))} (gatto, cane)")

# CIFAR-10 è 32×32, MobileNetV2 richiede minimo 96×96
# Usiamo GlobalAveragePooling e una CNN semplice per non dover fare resize

# Mostrare alcune immagini
clases_gc = ['Gatto', 'Cane']
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, (cls, label) in enumerate(zip([0, 1], ['Gatti', 'Cani'])):
    idx = np.where(y_tr_gc == cls)[0][:8]
    for ax, j in zip(axes[i], idx):
        ax.imshow(X_tr_gc[j])
        ax.axis('off')
    axes[i, 0].set_ylabel(label, rotation=90, va='center', fontsize=12)
plt.suptitle('CIFAR-10: Gatti vs Cani', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# CNN da zero per gatti vs cani (baseline)
tf.random.set_seed(42)
modelo_cifar = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid'),
], name='CNN_cifar_scratch')

modelo_cifar.compile('adam', 'binary_crossentropy', ['accuracy'])

h_cifar = modelo_cifar.fit(
    X_tr_gc, y_tr_gc, epochs=25, batch_size=64,
    validation_split=0.15,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=8, restore_best_weights=True)],
    verbose=1
)

_, acc_cifar = modelo_cifar.evaluate(X_te_gc, y_te_gc, verbose=0)
print(f"\nCNN da zero — Test accuracy: {acc_cifar:.4f}")

In [ ]:
# Transfer Learning: MobileNetV2 preaddestrata su ImageNet
# Dobbiamo ridimensionare a 96×96 minimo
X_tr_up = tf.image.resize(X_tr_gc, (96, 96)).numpy()
X_te_up = tf.image.resize(X_te_gc, (96, 96)).numpy()

# Caricare la base preaddestrata (senza il classificatore finale)
base_model = keras.applications.MobileNetV2(
    input_shape=(96, 96, 3),
    include_top=False,      # senza lo strato Dense finale
    weights='imagenet'      # pesi preaddestrati
)
base_model.trainable = False  # congelare tutti i pesi

print(f"MobileNetV2 base: {base_model.count_params():,} parametri (congelati)")

# Aggiungere il nostro classificatore sopra
entradas  = keras.Input(shape=(96, 96, 3))
x = keras.applications.mobilenet_v2.preprocess_input(entradas * 255)  # normalizzare per MV2
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salidas = layers.Dense(1, activation='sigmoid')(x)

modelo_tl = keras.Model(entradas, salidas, name='MobileNetV2_TL')
print(f"Parametri addestrabili: {sum(v.numpy().size for v in modelo_tl.trainable_variables):,}")

modelo_tl.compile('adam', 'binary_crossentropy', ['accuracy'])

h_tl = modelo_tl.fit(
    X_tr_up, y_tr_gc, epochs=10, batch_size=64,
    validation_split=0.15, verbose=1
)

_, acc_tl = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)
print(f"\nTransfer Learning (MobileNetV2) — Test accuracy: {acc_tl:.4f}")
print(f"CNN da zero                      — Test accuracy: {acc_cifar:.4f}")
print(f"Miglioramento con Transfer Learning: +{acc_tl-acc_cifar:.4f}")

---
## Pratica

### Esercizio 1 — CNN per Fashion MNIST

**Consegna:** Fashion MNIST è più impegnativo di MNIST — ideale per confrontare Dense vs CNN.

1. Carica e preelabora (28×28×1)
2. Architettura: 2 blocchi conv + classificatore Dense
3. Aggiungi Dropout e BatchNorm
4. Confronta con il modello Dense della lezione precedente
5. Quanti parametri ha ciascuno?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr_cnn = X_tr.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_te_cnn = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_tr_d   = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te_d   = X_te.reshape(-1, 784).astype('float32') / 255.0

# CNN
tf.random.set_seed(42)
m_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax'),
])
m_cnn.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_cnn.fit(X_tr_cnn, y_tr, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
_, acc_cnn = m_cnn.evaluate(X_te_cnn, y_te, verbose=0)

# Dense
tf.random.set_seed(42)
m_dense = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10,  activation='softmax'),
])
m_dense.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
m_dense.fit(X_tr_d, y_tr, epochs=15, batch_size=128, validation_split=0.1, verbose=0)
_, acc_dense = m_dense.evaluate(X_te_d, y_te, verbose=0)

print(f"CNN   — params: {m_cnn.count_params():,}, acc: {acc_cnn:.4f}")
print(f"Dense — params: {m_dense.count_params():,}, acc: {acc_dense:.4f}")
print(f"CNN usa {m_cnn.count_params()/m_dense.count_params():.1f}× meno parametri e ottiene un risultato migliore")
```
</details>

---
### Esercizio 2 — Visualizzare le attivazioni

**Consegna:** Cosa vede la rete in ogni strato?

1. Prendi il modello CNN di MNIST già addestrato
2. Usa `keras.Model(inputs=..., outputs=capa.output)` per estrarre le attivazioni di ogni blocco
3. Mostra i feature map degli strati: Conv1, Conv2, Conv3 e Conv4
4. Come cambia la 'rappresentazione' lungo gli strati?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import matplotlib.pyplot as plt
import numpy as np

# Strati conv del modello già addestrato
capas_conv = [l for l in modelo_cnn.layers if 'conv' in l.name]
print(f"Strati Conv: {[l.name for l in capas_conv]}")

# Immagine di esempio
img = X_te_cnn[0:1]  # (1, 28, 28, 1)
print(f"Classe reale: {y_te[0]}")

fig = plt.figure(figsize=(16, 10))
for row, capa in enumerate(capas_conv):
    m_interm = tf.keras.Model(inputs=modelo_cnn.input, outputs=capa.output)
    feat = m_interm.predict(img, verbose=0)  # (1, H, W, n_filtros)
    n_show = min(8, feat.shape[-1])

    for col in range(n_show):
        ax = fig.add_subplot(len(capas_conv), n_show, row*n_show + col + 1)
        ax.imshow(feat[0, :, :, col], cmap='viridis')
        if col == 0:
            ax.set_ylabel(capa.name[:10], fontsize=8, rotation=0, labelpad=40, va='center')
        ax.axis('off')

plt.suptitle(f'Feature map per strato per la cifra {y_te[0]}', fontsize=13)
plt.tight_layout(); plt.show()
print("→ Strato 1: bordi grezzi. Strato 2: forme. Strati profondi: pattern astratti.")
```
</details>

---
### Esercizio 3 — Data Augmentation

**Consegna:** Ampliare artificialmente il dataset con trasformazioni.

```python
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
```

1. Solo 2000 campioni di Fashion MNIST
2. Aggiungi data augmentation come primo blocco del modello
3. Confronta con lo stesso modello senza augmentation
4. Migliora la generalizzazione?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
N = 2000
X_tr_n = X_tr[:N].reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_tr_n = y_tr[:N]
X_te_n = X_te.reshape(-1, 28, 28, 1).astype('float32') / 255.0

def build_cnn(augmentation=False):
    capas = [layers.Input(shape=(28, 28, 1))]
    if augmentation:
        capas += [
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
        ]
    capas += [
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax'),
    ]
    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])
    return m

for nombre, aug in [('Senza augmentation', False), ('Con augmentation', True)]:
    tf.random.set_seed(42)
    m = build_cnn(aug)
    h = m.fit(X_tr_n, y_tr_n, epochs=30, batch_size=64,
              validation_split=0.15, verbose=0)
    _, acc_te = m.evaluate(X_te_n, y_te, verbose=0)
    acc_tr  = h.history['accuracy'][-1]
    acc_val = h.history['val_accuracy'][-1]
    print(f"{nombre}: tr={acc_tr:.4f}, val={acc_val:.4f}, te={acc_te:.4f}, gap={acc_tr-acc_val:.4f}")
```
</details>

---
### Esercizio 4 — Fine-tuning

**Consegna:** Il Transfer Learning con base congelata non sfrutta tutto il potenziale del modello preaddestrato. Il Fine-tuning scongela gli ultimi strati.

1. Usa il modello `modelo_tl` (MobileNetV2) già addestrato
2. Scongela gli ultimi 20 strati di `base_model`
3. Riaddestra con lr molto basso (1e-5) per 5 epoche
4. Migliora l'accuracy su gatti vs cani?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# Accuracy prima del fine-tuning (già addestrato sopra)
acc_antes = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)[1]
print(f"Prima del fine-tuning: {acc_antes:.4f}")

# Scongelare gli ultimi 20 strati
base_model.trainable = True
for capa in base_model.layers[:-20]:
    capa.trainable = False

n_train = sum(v.numpy().size for v in modelo_tl.trainable_variables)
print(f"Parametri addestrabili dopo aver scongelato gli ultimi 20 strati: {n_train:,}")

# Ricompilare con lr molto basso (indispensabile nel fine-tuning)
modelo_tl.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # lr molto basso: non distruggere i pesi
    loss='binary_crossentropy',
    metrics=['accuracy']
)

h_ft = modelo_tl.fit(
    X_tr_up, y_tr_gc, epochs=5, batch_size=32,
    validation_split=0.15, verbose=1
)

acc_despues = modelo_tl.evaluate(X_te_up, y_te_gc, verbose=0)[1]
print(f"\nPrima del fine-tuning:  {acc_antes:.4f}")
print(f"Dopo il fine-tuning:    {acc_despues:.4f}")
print(f"Miglioramento: +{acc_despues-acc_antes:.4f}")
```
</details>

---
### Esercizio 5 (sfida) — GradCAM: dove guarda la rete?

**Consegna:** GradCAM (Gradient-weighted Class Activation Mapping) mostra quale regione dell'immagine influenza di più la predizione.

1. Implementa una funzione `gradcam(model, img, clase_idx, nombre_capa)` che:
   - Calcoli i gradienti della classe rispetto ai feature map dell'ultimo strato conv
   - Generi la mappa di calore
   - Sovrapponga la mappa all'immagine originale
2. Applicala a 5 immagini di MNIST
3. In quale parte della cifra si concentra la rete per predire ogni classe?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def gradcam(model, img, clase_idx, nombre_capa):
    """img: (1, H, W, C) normalizzata"""
    capa = model.get_layer(nombre_capa)
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[capa.output, model.output]
    )
    with tf.GradientTape() as tape:
        feat_maps, preds = grad_model(img)
        score = preds[0, clase_idx]

    grads   = tape.gradient(score, feat_maps)      # (1, H, W, n_filtros)
    pooled  = tf.reduce_mean(grads, axis=(0,1,2))  # (n_filtros,)
    heatmap = tf.reduce_mean(feat_maps[0] * pooled, axis=-1)  # (H, W)
    heatmap = tf.nn.relu(heatmap).numpy()
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    return heatmap

# Ultimo strato conv del modello MNIST
capas_conv = [l.name for l in modelo_cnn.layers if 'conv' in l.name]
ultima_conv = capas_conv[-1]
print(f"Usando lo strato: {ultima_conv}")

# 5 immagini di classi diverse
n_clases = 5
fig, axes = plt.subplots(n_clases, 3, figsize=(10, 3*n_clases))

for i, clase in enumerate(range(n_clases)):
    idx = np.where(y_te == clase)[0][0]
    img = X_te_cnn[idx:idx+1]

    heatmap = gradcam(modelo_cnn, img, clase, ultima_conv)
    heatmap_resized = tf.image.resize(heatmap[..., None], (28, 28)).numpy()[:,:,0]

    # Originale
    axes[i,0].imshow(img[0,:,:,0], cmap='gray')
    axes[i,0].set_title(f'Cifra reale: {clase}'); axes[i,0].axis('off')

    # Heatmap
    axes[i,1].imshow(heatmap_resized, cmap='hot')
    axes[i,1].set_title('GradCAM'); axes[i,1].axis('off')

    # Sovrapposizione
    img_rgb = np.stack([img[0,:,:,0]]*3, axis=-1)
    hm_rgb  = plt.cm.jet(heatmap_resized)[:,:,:3]
    overlay = 0.6 * img_rgb + 0.4 * hm_rgb
    overlay = np.clip(overlay, 0, 1)
    axes[i,2].imshow(overlay)
    axes[i,2].set_title('Sovrapposizione'); axes[i,2].axis('off')

plt.suptitle('GradCAM: dove guarda la rete per predire ogni cifra', fontsize=13)
plt.tight_layout(); plt.show()
```
</details>

---
## Riepilogo

```python
# CNN tipica in Keras
modelo = keras.Sequential([
    layers.Input(shape=(H, W, C)),   # C=1 scala di grigi, C=3 RGB

    # Blocco convoluzionale
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(2),           # ÷2 nello spazio
    layers.Dropout(0.25),

    # Altri blocchi...

    # Classificatore
    layers.Flatten(),                 # o GlobalAveragePooling2D()
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(n_clases, activation='softmax'),
])

# Transfer Learning
base = keras.applications.MobileNetV2(include_top=False, weights='imagenet')
base.trainable = False
# ... aggiungere il classificatore sopra
```

| Strato | Funzione | Parametri |
|------|---------|------------|
| Conv2D(f, k) | Rilevare pattern locali | k²·C·f + f |
| MaxPooling2D(2) | Ridurre la risoluzione ÷2 | 0 |
| GlobalAveragePooling2D | Riassumere il feature map in 1 valore | 0 |
| Flatten | Appiattire a 1D | 0 |

## Cosa viene nella prossima lezione?
Nella **Lezione 8** impareremo **RNN e Transformer** — architetture per sequenze (testo, audio, serie temporali) con LSTM, GRU e il meccanismo di attenzione.